### (more) Effective Energy Shift Algorithm for Energy Storage System Analysis

This notebook is intended as an interactive playground to foster the understanding of the Effective Energy Shift (EfES) algorithm as well as the more Effective Energy Shift (mEfES).

In [1]:
import numpy as np
import pandas as pd
from typing import Any

#### Chronological reference algorithm

The chronological reference algorithm performs a simple sweep over the provided power series and uses the same greedy policy as implied in the EfES algorithm. It utilizes a steady-state-detection based on the repeating state-of-charge trajectory of the ESS.

The reference result caluculation can be enabled in the interactive vizualisation and you get the best experience, when you disable all other plots. 

#### Import modules and define functions for visualization

In [3]:
import ipywidgets as widgets
from ipywidgets import interactive_output
import copy

import mefes
import efes
from efes_core import EfesInput, QueryInput, Results, run_chrono_ref_alg


def get_example_input(impl: str = 'mefes') -> EfesInput:
    """Get the example input for a specific implementation: 'efes' or 'mefes'"""
    match impl.lower():
        case 'efes':
            power_generation, power_demand =  efes.examples.build_example_time_series()
        case 'mefes':
            power_generation, power_demand =  mefes.examples.build_example_time_series()
        case _:
            raise AttributeError(f'Implementation "{impl}" not known. Use "efes" or "mefes"')

    return EfesInput(
        power_generation=power_generation,
        power_demand=power_demand,
        delta_time_step=1.
    )

    
def create_tabel_from_results(*results: Any) -> pd.DataFrame:
    """
    Build a comparison table from multiple Results objects.

    Expected structure per result:
        result.analysis_results.used_method -> str
        result.query_results[0].capacity -> array-like
        result.query_results[0].energy_additional -> array-like

    Output columns follow the same order as the passed results, with delta columns
    inserted between adjacent result columns.

    Example column order for:
        create_tabel_from_results(ref_res, efes_res, mefes_res)

    becomes:
        capacity,
        Chrono. ref.,
        delta(Chrono. ref.-EfES),
        EfES,
        delta(EfES-mEfES),
        mEfES
    """
    if not results:
        raise ValueError("At least one result object must be provided.")

    # Extract first query result from each implementation
    extracted = []
    for result in results:
        impl_name = getattr(result.analysis_results, "used_method", None)
        if impl_name is None:
            raise AttributeError(f"Missing 'used_method' field on result object: {result!r}")

        query_results = getattr(result, "query_results", None)
        if not query_results:
            raise ValueError(f"Result object '{impl_name}' has no query_results.")

        qr = query_results[0]

        capacity = np.asarray(getattr(qr, "capacity"))
        energy_additional = np.asarray(getattr(qr, "energy_additional"))

        extracted.append(
            {
                "impl": impl_name,
                "capacity": capacity,
                "energy_additional": energy_additional,
            }
        )

    # Validate that all capacities are identical
    ref_capacity = extracted[0]["capacity"]
    for item in extracted[1:]:
        if not np.array_equal(ref_capacity, item["capacity"]):
            raise ValueError(
                f"Capacity mismatch between '{extracted[0]['impl']}' and '{item['impl']}'."
            )

    # Build dataframe in requested order
    data: dict[str, np.ndarray] = {
        "capacity": ref_capacity
    }

    for i, item in enumerate(extracted):
        impl = item["impl"]
        values = item["energy_additional"]

        data[impl] = values

        if i < len(extracted) - 1:
            next_item = extracted[i + 1]
            next_impl = next_item["impl"]
            delta_col = f"delta({impl}-{next_impl})"
            data[delta_col] = values - next_item["energy_additional"]

    return pd.DataFrame(data)


    
from efes.adapters.scenarios.plotting import plot_energy_packets
from efes_core.adapters.scenarios.plotting import plot_input, plot_results

style_widgets = dict(description_width='initial', handle_color='lightblue')

def create_intermediate_result_player(intermediate_results, efficiency_discharging):
    if len(intermediate_results) == 0:
        return 
        
    widget_intermediate_result_play = widgets.Play(
        value=len(intermediate_results)-1,
        min=0,
        max=len(intermediate_results)-1,
        step=1,
        interval=1000,
        description="Press play",
        disabled=False
    )
    widget_intermediate_result_index=widgets.IntSlider(
        value=len(intermediate_results)-1,
        min=0,
        max=len(intermediate_results)-1,
        step=1,
        style = style_widgets,
        continuous_update=True,
        orientation='horizontal',
        readout=True
    )
    widgets.jslink((widget_intermediate_result_play, 'value'), (widget_intermediate_result_index, 'value'))

    def show_intermediate_result(intermediate_result_index):
        plot_energy_packets(*(intermediate_results[intermediate_result_index]), efficiency_discharging)
        
        
    ui = widgets.VBox([widgets.Label('Use the player or slider to scroll through the performed steps:'), widgets.HBox([widget_intermediate_result_play,widget_intermediate_result_index])])

    out = interactive_output(show_intermediate_result, dict(intermediate_result_index=widget_intermediate_result_index))
    display(ui,out)

    
current_results = None

def create_widget(widget_type, description, initial_value, **kwargs):
    if widget_type == 'Checkbox':
        return widgets.Checkbox(
            value=initial_value,
            description=description,
            disabled=False,
            indent=False,
            style = style_widgets,
            **kwargs
        )
    if widget_type == 'FloatText':
        return widgets.FloatText(
            value=initial_value,
            description=description,
            style = style_widgets,
            disabled=False,
            **kwargs
        )
    if widget_type == 'FloatSlider':
        return widgets.FloatSlider(
            value=initial_value,
            description=description,
            style = style_widgets,
            continuous_update=True,
            readout=True,
            readout_format='.2f',
            **kwargs
        )
    if widget_type == 'Dropdown':
        return widgets.Dropdown(            
            value=initial_value,
            description=description,
            style = style_widgets,
            **kwargs
        )

def create_ui_from_descr(ui_descr, widget_dict):
    v_box_entries = []
    for v_box_descr in ui_descr:
        if isinstance(v_box_descr, str):
            v_box_entries.append(widget_dict[v_box_descr])
        else:
            h_box_entries = []
            for h_box_descr in v_box_descr:
                h_box_entries.append(widget_dict[h_box_descr])
            v_box_entries.append(widgets.HBox(h_box_entries))
    return widgets.VBox(v_box_entries)
    

from efes_core.domain.ports import ObserverPort, AlgorithmState
from efes_core.domain.enums import PacketType
from efes.domain.models import EfesState, Phase
from mefes.domain.models import MefesState

def mefes_to_efes(state: MefesState) -> EfesState:
    efes_state = EfesState()
    efes_state.step = state.step
    efes_state.phases = []
    efes_state.mask = np.ones((2, len(state.phase_pairs)), dtype=bool)
    for n, phase_pair in enumerate(state.phase_pairs):        
        phase = Phase(energy_excess_initial=[1], energy_deficit_initial=[1])
        phase.id = n
        phase.starts_excess = []
        phase.starts_deficit = []
        phase.energy_excess = []
        phase.energy_deficit = []
        phase.excess_balanced = []
        phase.deficit_balanced = []
        phase.excess_ids = []
        
        for ep in phase_pair.energy_packets[PacketType.BALANCED]:
            phase.starts_excess.append(ep.capacity)
            phase.energy_excess.append(ep.energy)
            phase.excess_balanced.append(True)
            phase.excess_ids.append(n)
            phase.starts_deficit.append(ep.capacity)
            phase.energy_deficit.append(ep.energy)
            phase.deficit_balanced.append(True)
        
        for ep in phase_pair.energy_packets[PacketType.EXCESS]:
            phase.starts_excess.append(ep.capacity)
            phase.energy_excess.append(ep.energy)
            phase.excess_balanced.append(False)            
            phase.excess_ids.append(n)
            efes_state.mask[0][n] = True

        for ep in phase_pair.energy_packets[PacketType.DEFICIT]:
            phase.starts_deficit.append(ep.capacity)
            phase.energy_deficit.append(ep.energy)
            phase.deficit_balanced.append(False)            
            efes_state.mask[1][n] = True

        phase.starts_excess = np.array(phase.starts_excess)
        phase.starts_deficit = np.array(phase.starts_deficit)
        phase.energy_excess = np.array(phase.energy_excess)
        phase.energy_deficit = np.array(phase.energy_deficit)
        phase.excess_balanced = np.array(phase.excess_balanced)
        phase.deficit_balanced = np.array(phase.deficit_balanced)
        phase.excess_ids = np.array(phase.excess_ids)
        
        efes_state.phases.append(phase)
        
    return efes_state   
    
    
class Recorder(ObserverPort):
    def __init__(self):
        global intermediate_results
        intermediate_results = []
        
    def on_step(self, state: AlgorithmState) -> bool:
        
        global intermediate_results, current_results

        # Nothing meaningful to plot before phases exist.
        if isinstance(state, EfesState):
            intermediate_results.append((state.step, copy.deepcopy(state.phases), copy.deepcopy(state.mask)))
        
        if isinstance(state, MefesState):
            efes_state = mefes_to_efes(state)
            intermediate_results.append((efes_state.step, copy.deepcopy(efes_state.phases), copy.deepcopy(efes_state.mask)))
    

def create_ui(power_generation, power_demand, delta_time_step):
    energy_generation_original = power_generation.sum()
    power_generation_normalized = power_generation/energy_generation_original

    energy_demand_original = power_demand.sum()
    power_demand_normalized = power_demand/energy_demand_original

    intermediate_results = []

    implementations = ['mEfES', 'EfES']
    
    def update(
        implementation,
        energy_demand, 
        energy_generation, 
        efficiency_charging, 
        efficiency_discharging, 
        efficiency_direct_usage, 
        add_plot_input, 
        add_plot_intermediate_steps, 
        add_plot_results,
        calc_ref_results,
    ):
        global current_results, intermediate_results
        power_generation_rescaled = energy_generation*power_generation_normalized
        power_demand_rescaled = energy_demand*power_demand_normalized
        
        efes_input = EfesInput(
            power_generation=power_generation_rescaled,
            power_demand=power_demand_rescaled,
            delta_time_step=1.,
            power_max_charging=np.inf,
            efficiency_direct_usage=efficiency_direct_usage,
            efficiency_charging=efficiency_charging,
            efficiency_discharging=efficiency_discharging,
        )
        query_input = None
        observer = Recorder()

        match implementations[implementation]:
            case 'EfES':
                print(f'Running EfES...')
                current_results = efes.run(
                    efes_input=efes_input,
                    query_input=query_input,
                    observer=observer,
                )      
            case 'mEfES':
                print(f'Running mEfES...')
                current_results = mefes.run(
                    efes_input=efes_input,
                    query_input=query_input,
                    observer=observer,
                )
            case _:
                current_results = None
                return
                
        
        if add_plot_input:
            plot_input(current_results)
        if add_plot_intermediate_steps:
            create_intermediate_result_player(intermediate_results, efficiency_discharging)
        if add_plot_results:
            plot_results(current_results)
        if calc_ref_results:
            from efes_core.domain.services import run_dimensioning_query_for_target_capacity
            
            query_input = QueryInput(
                capacity_target=current_results.analysis_results.capacity,
            )

            print(f'Running chrono. ref. ...')
            ref_res = run_chrono_ref_alg(
                efes_input=efes_input,
                query_input=query_input,
                capacity_bounds=(0, 2*query_input.capacity_target[-1]),
                n_samples=100000,
            )

            current_results.query_results = [run_dimensioning_query_for_target_capacity(
                current_results.analysis_results, ref_res.query_results[0].query_input.capacity_target)]
            
            
            chrono_df = create_tabel_from_results(ref_res, current_results)
            chrono_df.columns = [
                'capacity',
                'chronoRef',
                'delta',
                implementations[implementation],
            ]
            
            display(chrono_df)
            
    widget_dict = dict(
        implementation              = create_widget(widget_type='Dropdown', description='implementation:', initial_value=0, options=[(impl, n) for (n, impl) in enumerate(implementations)]),        
        energy_demand               = create_widget(widget_type='FloatText', description='energy demand:', initial_value=energy_demand_original),
        energy_generation           = create_widget(widget_type='FloatText', description='energy generation:', initial_value=energy_generation_original),
        efficiency_charging         = create_widget(widget_type='FloatSlider', description='efficiency charging:', initial_value=1.0, min=0.1, max=1.0, step=0.01, orientation='horizontal' ),
        efficiency_discharging      = create_widget(widget_type='FloatSlider', description='efficiency discharging:', initial_value=1.0, min=0.1, max=1.0, step=0.01, orientation='horizontal' ),
        efficiency_direct_usage     = create_widget(widget_type='FloatSlider', description='efficiency direct usage:', initial_value=1.0, min=0.1, max=1.0, step=0.01, orientation='horizontal' ),
        add_plot_input              = create_widget(widget_type='Checkbox', description='Plot input', initial_value=False),        
        add_plot_intermediate_steps = create_widget(widget_type='Checkbox', description='Plot intermediate steps', initial_value=False),
        add_plot_results            = create_widget(widget_type='Checkbox', description='Plot results', initial_value=True),
        calc_ref_results            = create_widget(widget_type='Checkbox', description='Chronological calculation', initial_value=False),
    )

    ui_descr = ['implementation', ['energy_demand', 'energy_generation'], ['efficiency_direct_usage', 'efficiency_charging', 'efficiency_discharging'], 'add_plot_input', 'add_plot_intermediate_steps', 'add_plot_results', 'calc_ref_results']
    ui = create_ui_from_descr(ui_descr, widget_dict)

    out = interactive_output(update, widget_dict)
    
    return ui, out

#### Interactive example from the publication

In [4]:
input_example = get_example_input(impl = 'mefes')
ui, out = create_ui(input_example.power_generation, input_example.power_demand, input_example.delta_time_step)
display(ui,out)

Output()